Notebook 6: Lyric Availability Assessment

The user survey found that 61% of respondents weight lyrical content above musical
feel when selecting music for a mood, suggesting a multimodal extension combining
acoustic features with lyric sentiment. This notebook tests whether such an extension
is feasible with DEAM by measuring lyric availability for a random sample of 30 tracks.

In [2]:
!pip install -q lyricsgenius

import os, time
import pandas as pd
import lyricsgenius
from getpass import getpass

from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/Dissertation/data/processed"
song_pool = pd.read_csv(os.path.join(DATA_PATH, "song_pool_predictions.csv"))

GENIUS_TOKEN = getpass("Genius token: ")
genius = lyricsgenius.Genius(GENIUS_TOKEN)
genius.verbose = False
genius.remove_section_headers = True
genius.skip_non_songs = True
genius.timeout = 10

sample = song_pool.sample(30, random_state=42)

found, missing = [], []
for r in sample.itertuples():
    try:
        s = genius.search_song(r.title, r.Artist)
        if s and s.lyrics and len(s.lyrics) > 100:
            found.append((r.song_id, r.title, r.Artist))
        else:
            missing.append((r.title, r.Artist))
    except Exception:
        missing.append((r.title, r.Artist))
    time.sleep(0.5)

print(f"Found: {len(found)}/30")
for f in found[:5]:
    print(" ", f[1], "-", f[2])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Genius token: ··········
Found: 0/30


In [4]:
import requests, time

def get_lyrics_ovh(artist, title):
    url = f"https://api.lyrics.ovh/v1/{artist}/{title}"
    try:
        r = requests.get(url, timeout=8)
        if r.status_code == 200:
            return r.json().get("lyrics", "")
    except Exception:
        pass
    return None

found, missing = [], []
for r in sample.itertuples():
    lyr = get_lyrics_ovh(r.Artist, r.title)
    if lyr and len(lyr) > 100:
        found.append((r.song_id, r.title, r.Artist))
    else:
        missing.append((r.title, r.Artist))
    time.sleep(0.4)

print(f"Found: {len(found)}/30")
for f in found[:5]:
    print(" ", f[1], "-", f[2])

Found: 1/30
  Rose Room - Jeremy Cohen & Matt Munisteri
